# 🚀 MiniGPT: Pro Scaling Challenge (124M Parameters / 10GB Data)

Welcome to the high-scale training notebook. This setup is optimized to reach GPT-2 Small intelligence on free hardware.

### ⚠️ Essential Advice for the 10GB Run:
1. **Hardware**: You MUST use a **T4 GPU** (Standard on Free Colab).
2. **Timeline**: This run will take approximately **40-50 hours**. Use the 'Resume' feature to split it over several days.
3. **Persistence**: We use Google Drive to save checkpoints so you can resume if Colab disconnects.
4. **Batch Size**: We use **Gradient Accumulation** to simulate a large batch size of 64+ without crashing the GPU.

## Step 0: Mount Google Drive
This ensures your checkpoints are safe even if the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a project folder in your Drive
import os
os.makedirs('/content/drive/MyDrive/MiniGPT_Checkpoints', exist_ok=True)

## Step 1: Clone & Setup

In [ ]:
if not os.path.exists('MiniGPT-from-Scratch'):
    !git clone https://github.com/mrshibly/MiniGPT-from-Scratch.git

%cd MiniGPT-from-Scratch
!pip install -r requirements.txt

# Link the checkpoints folder to your Google Drive
!rm -rf checkpoints
!ln -s /content/drive/MyDrive/MiniGPT_Checkpoints checkpoints

## Step 2: Download & Preprocess (10GB Scaling)
**Note**: Downloading and tokenizing 10GB will take ~1 hour.

In [ ]:
# Set scaling limit to 10GB (Swapping the 500MB default for 10GB)
!sed -i 's/max_bytes = 500/max_bytes = 10 * 1024/' src/datasets/download_fineweb.py

!python src/datasets/download_fineweb.py
!python src/datasets/clean_text.py
!python src/tokenizer/train_tokenizer.py
!python src/datasets/prepare_data.py

## Step 3: Launch Pro Training (124M Params)
The model will automatically load `ckpt.pt` from your Drive if it exists.

In [ ]:
# Configure for the 124M 'Stretch' run
!sed -i 's/config = MiniGPTConfig.standard()/config = MiniGPTConfig.stretch()/' src/train/train.py
!sed -i 's/max_steps = 10000/max_steps = 100000/' src/train/train.py

# Optimizing for T4 GPU (Effective Batch Size = 8 * 8 = 64)
!sed -i 's/batch_size = 8/batch_size = 8/' src/train/train.py
!sed -i 's/grad_accum_steps = 4/grad_accum_steps = 8/' src/train/train.py

!python src/train/train.py